# Analisis del benchmark

Aquest notebook servira per analitzar els resultats del benchmark de colors. Intento que aqui nomes hi hagi els passos generals, i que les funcions estiguin a `scripts/`.

Si el dataset encara no existeix, primer cal executar el notebook de `recollida-dades`.

## Configuracion

Netegem l'entorn, carreguem llibreries i definim rutes relatives. Uso `..` per apuntar a les dades generades a la fase anterior.

Tambee deixo que cada cel.la imprimeixi el temps que ha trigat. Va be per detectar si alguna part es queda massa lenta quan la mostra creixi.


In [ ]:
cell_start <- Sys.time()

format_duration <- function(seconds) {
  seconds <- max(0, as.numeric(seconds))
  if (seconds < 60) return(sprintf("%.1fs", seconds))
  minutes <- floor(seconds / 60)
  remaining_seconds <- seconds %% 60
  if (minutes < 60) return(sprintf("%dm %04.1fs", minutes, remaining_seconds))
  hours <- floor(minutes / 60)
  remaining_minutes <- minutes %% 60
  sprintf("%dh %dm %04.1fs", hours, remaining_minutes, remaining_seconds)
}

print_cell_time <- function(start_time) {
  elapsed <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))
  cat("Temps cel.la:", format_duration(elapsed), "\n")
}

rm(list = setdiff(ls(), c("format_duration", "print_cell_time", "cell_start")))

suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(magick))

analysis_dir <- if (dir.exists("analisis")) file.path(".", "analisis") else "."
data_root <- if (dir.exists("recollida-dades")) "." else ".."

csv_dir <- file.path(data_root, "recollida-dades", "csv")
images_dir <- file.path(data_root, "recollida-dades", "images")
scripts_dir <- file.path(analysis_dir, "scripts")

print_cell_time(cell_start)


## Carrega de scripts

Carreguem les funcions d'analisi. Si una funcio no existeix, parem el notebook per no interpretar resultats malament.

In [ ]:
cell_start <- Sys.time()

script_files <- list.files(scripts_dir, pattern = "\\.R$", full.names = TRUE)
walk(script_files, source)

if (!exists("clean_dataset")) stop("Error: scripts no cargados correctamente")
if (!exists("extract_main_color")) stop("Error: scripts no cargados correctamente")
if (!exists("evaluate_predictions")) stop("Error: scripts no cargados correctamente")

print_cell_time(cell_start)


## Carga de datos

Intentamos cargar el CSV final (`sample-colors.csv`). Si todavia no existe porque no hemos lanzado la API, cargamos `input_image_sample.csv` para poder revisar la muestra de colores.

In [ ]:
cell_start <- Sys.time()

final_path <- file.path(csv_dir, "sample-colors.csv")
input_path <- file.path(csv_dir, "input_image_sample.csv")

if (file.exists(final_path)) {
  raw_data <- read_csv(final_path, show_col_types = FALSE)
} else if (file.exists(input_path)) {
  warning("Todavia no existe sample-colors.csv. Cargo input_image_sample.csv para revisar la muestra inicial.")
  raw_data <- read_csv(input_path, show_col_types = FALSE)
} else {
  warning("No se han encontrado CSVs. Ejecuta primero recollida-dades/dades.ipynb")
  raw_data <- tibble(image_name = character(), hex = character(), r = integer(), g = integer(), b = integer(), chroma = double())
}

print(raw_data)
print_cell_time(cell_start)


## Procesamiento

Limpiamos el dataset con la funcion del script. Mas adelante aqui se podrian calcular predicciones o leer resultados de modelos.

In [ ]:
cell_start <- Sys.time()

clean_data <- clean_dataset(raw_data)

print(clean_data)
print_cell_time(cell_start)


## Resultados

De momento dejamos preparada la evaluacion. La evaluacion completa funcionara cuando tengamos las respuestas de los modelos y exista `sample-colors.csv`.

In [ ]:
cell_start <- Sys.time()

example_predictions <- tibble(
  expected_color = c("red", "blue"),
  predicted_color = c("red", "green")
)

print(evaluate_predictions(example_predictions))
print_cell_time(cell_start)
